### 환경 업그레이드
선정한 모델이 파이토치 2.6이상에서 작동하기에 업그레이드를 하였음

In [2]:
# 1. 최신 PyTorch 2.6.0 이상 + CUDA 12.4 빌드 설치
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install --upgrade transformers

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 6.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 40.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.4/76

### 모델 테스트
pipeline을 통해서 간단한 테스트를 한다. 이 과정에서 다운로드도 이뤄지고 (~/.cache/Huggingface/hub) 바로 사용할 수 있는 상태가 된다.

In [6]:
from transformers import pipeline

sentiment_model = pipeline(model="WhitePeak/bert-base-cased-Korean-sentiment")
sentiment_model("매우 좋아")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.8098212480545044}]

### 프로젝트 폴더 구조
```
my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt
```

### 스키마 설계 Pydantic 검증

In [13]:
%%writefile app/schemas.py

from pydantic import BaseModel, Field

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000, description="분석할 평가 문장")

    model_config = {
        "json_schema_extra": {
            "examples":[
                {
                    "text": "매우 좋아"
                }
            ]
        }
    }
    
class PredictResponse(BaseModel):
    success: bool = Field(..., description="요청 처리 성공 여부")
    label: str = Field(..., description="예측된 감정 (긍정/부정)")
    confidence: float = Field(..., ge=0, le=1, description="모델의 예측확신도")

    model_config = {
        "json_schema_extra": {
            "examples":[
                {
                    "success": True,
                    "label": "긍정",
                    "confidence": 0.9985
                }
            ]
        }
    }

Overwriting app/schemas.py


### 모델 로드 및 추론 함수 설계

In [18]:
%%writefile app/model_service.py

import torch
from transformers import pipeline


def load_model():
    # 0: 첫 번째 GPU, -1: CPU 사용
    device = 0 if torch.cuda.is_available() else -1
    
    return = pipeline(
        "text-classification",
        model="WhitePeak/bert-base-cased-Korean-sentiment",
        model_kwargs={"weights_only": False},
        device=device
    )

def predict(model, text: str) -> dict:
    raw_result = model(text)[0]

    label_map = {"LABEL_1": "긍정", "LABEL_0": "부정"}
    display_label = label_map.get(raw_result["label"], raw_result["label"])
    
    return {
        "label": display_label,
        "confidence": round(float(raw_result["score"]), 4)
    }

Overwriting app/model_service.py


### FastAPI 구현

In [27]:
%%writefile app/main.py

import asyncio
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, Depends, HTTPException
from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware

logger = setup_logger(__name__)

app = FastAPI(
    title="한국어 특화 감성 분석 판독기",
    description="한국어 문장을 읽고 긍정/부정 판단하는 API.",
    version="1.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

executor = ThreadPoolExecutor(
    max_workers=4,
    thread_name_prefix="sentiment_inference"
)
model = None

@app.on_event("startup")
async def startup():
    global model
    logger.info("감성 분석 모델 로드 중")
    model = load_model()
    logger.info("모델 로드 완료")

@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if model is not None else "loading...",
        "loaded": model is not None,
    }

@app.post("/predict", response_model=PredictResponse, tags=["Prediction"])
async def predict_sentiment(
    request: PredictRequest,
    user: str = Depends(verify_api_key)
    ):
    if model is None:
        raise HTTPException(status_code=503, detail="모델이 아직 준비되지 않았습니다.")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            executor,
            predict,
            model,
            request.text
        )
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

    return {
        "success": True,
        "label": result["label"],
        "confidence": result["confidence"]
    }

Overwriting app/main.py


### Streamlit API 구현

In [29]:
%%writefile frontend/app.py

import streamlit as st
import requests

st.set_page_config(
    page_title="한국어 감성 분석기",
    page_icon="😊",
    layout="centered",
)

API_BASE = "http://localhost:8000"

def call_api(url, json_data=None, method="post"):
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error(" 서버에 연결할 수 없습니다. FastAPI 서버가 실행중인지 확인하세요")
        return None
    except requests.exceptions.Timeout:
        st.warning(" 응답 시간 초과. 잠시 후 다시 시도하세요")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f" 서버 에러 (HTTP {e.response.status_code})")
    except Exception as e:
        st.error(f" 오류: {type(e).__name__}")
        return None

st.title("😊 한국어 감성 분석 판독")
st.write("문장을 입력하면 긍정인지 부정인지 분석해 드립니다.")
    
with st.sidebar:
    st.header("🔐API 보안 설정")
    api_key = st.text_input("API Key", type="password", value="test-key-001")
    st.divider()

    health = call_api(f"{API_BASE}/health", method="get")
    if health and "healthy" in health.get("status", ""):
        st.success("🟢 서버 연결됨")
        server_ok = True
    else:
        st.error("🔴 서버 연결 실패")
        server_ok = False

    st.divider()
    st.caption("Sentiment Predictor Dashboard v1.0")

with st.container():
    user_input = st.text_area(
        "분석할 문장을 입력하세요",
        placeholder="예: 오늘 날씨도 좋고 공부도 잘 돼서 정말 행복해!",
        height=150
    )

if st.button("감성 분석 시작", use_container_width=True):
    if not user_input.strip():
        st.warning("문장을 입력해 주세요.")
    else:
        with st.spinner("AI가 문장을 읽고 분석 중..."):
            try:
                response = requests.post(
                    f"{API_BASE}/predict",
                    json={"text": user_input},
                    headers={"X-API-Key": api_key},
                    timeout=30
                )

                if response.status_code == 200:
                    result = response.json()

                    st.divider()
                    st.subheader("📊분석 결과")

                    label = result["label"]
                    confidence = result["confidence"] * 100

                    col1, col2 = st.columns(2)

                    with col1:
                        if label == "긍정":
                            st.success(f"### 결과: {label} 😊")
                        else:
                            st.error(f"### 결과: {label} 😢")
                            
                    with col2:
                        st.metric("확신도", f"{confidence:.2f}%")
                        st.progress(result["confidence"])

                elif response.status_code == 401:
                    st.error("API Key가 올바르지 않습니다.")
                elif response.status_code == 503:
                    st.warning("모델이 로드 중입니다. 잠시 후 다시 시도해 주세요.")
                else:
                    st.error(f" 에러 발생: {response.json().get('detail', '알 수 없는 에러')}")

            except requests.exceptions.ConnectionError:
                st.error("서버가 실행 중이 아닙니다. 포트 8000을 확인하세요")
            except Exception as e:
                st.error(f"예상치 못한 오류: {str(e)}")

st.divider()
st.caption("Model: WhitePeak/bert-base-cased-Korean-sentiment")

Overwriting frontend/app.py


### 테스트

In [26]:
import requests

API_URL = "http://localhost:8000"
HEADERS = {"X-API-Key": "test-key-001"}

# health check
print(requests.get(f"{API_URL}/health").json())

# 추론 테스트 — 본인의 입력에 맞게 수정하세요
response = requests.post(
    f"{API_URL}/predict",
    json={"text": "행복해"},   # ← 수정
    headers=HEADERS,
)
print(f"상태: {response.status_code}")
print(f"결과: {response.json()}")

{'status': 'healthy', 'loaded': True}
상태: 200
결과: {'success': True, 'label': '긍정', 'confidence': 0.8987}


### ✅ Day 8 최종 체크포인트

#### Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?
> - 최소 1자 이상의 text가 반드시 있어야 하며 5천자 이하의 길이 제한
> - text 하나만 규정 하고 있기 때문에, 의도된 공격 코드를 방지
    
#### Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?
> - 인증 요구 없이 실행 됩니다.
> - 서버 보안이 취약해집니다. 
> - 악의적인 요청으로 서버 자원 사용량 증가
> - 서버 과부하로 이용 불가
> - 모델 복제 가능성
    
#### Q3. run_in_executor를 사용한 이유는 무엇입니까?
> - 비동기함수 안에서 모델 추론, CPU 바운드등의 무거운 동기 작업을 수행할 때, 이를 별도의 스레드풀에 할당하여 실행합니다. 이로써 이벤트 루프는 블로킹이 방지 되며 다른 요청을 계속해서 처리 할 수 있습니다.
    
#### Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?
> - 4, 5, 7등 실습 코드가 많이 있는 곳을 주로 참고 했습니다. 핵심 개념은 오늘 day8 교안으로도 충분히 리마인드 되었지만, 막상 코드가 생각이 안나서 실제 코드를 많이 보았습니다.
    
#### Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?
> - 지금은 무상태 지향이지만 결국엔 DB에 상태를 저장해야 최소한의 회원 관리 및 구분을 할 수 있을 것.
> - Docker 도입


### 구현 과정 & 결과
[day8.md](day8.md)

### 회고

스스로 돌아보기:
#### Day 1~7 교안 없이 코드를 작성할 수 있었는가?
> 아니요. 단시간에 배운 것이 많아서 다시 찾지 않고는 코드를 작성할 수는 없었습니다. 그래도 1~7일까지 매일 코드를 매우 면밀히 이해하려고 노력했기 때문에, 다시 보면 무엇을 하는 건지 어느정도 이해가 되어서 좋았습니다.
#### 어떤 부분에서 교안을 다시 찾아봤는가?
> - schema: 정확한 Pydantic 문법, Swagger UI 설명, 기본값 표시하는 법, 클래스가 요청/응답 두개면 되는지 확인
> - model 설계: 모델 부르는 부분을 찾아서 비교해 보았다.
> - backend API: logger, middleware, error handler 등 정확한 문법 확인
> - frontend API: 모델 로드 방식 확인
#### 다음에 다시 만든다면 무엇을 다르게 하겠는가?
> 모델 이름만 바꾸면 되는 것이 아닌 다른 설계도 모두 바뀌어야 한다. 모델의 태스크에 따라 크게 변하기 때문에, 다른 태스크도 연습해 보아야 한다.